# 03 · Leakage and the feature set (Team 10, Deliverable 3)

**Prediction:** passengers on one route in one month, made at a forecast origin T, for every
month from T to T+23. **Cutoff:** T. An input is allowed only if it is known at T *for all 24
horizons*, not just for next month.

## What is known at the cutoff?

| Input | months 1–12 after T | months 13–24 after T | Decision |
|---|---|---|---|
| `OD_REV_USD` for the same month | not known: it is paid by the passengers being predicted | not known | **drop** |
| Passengers last month (lag 1), rolling means | known only for month 1 | not known | **not used** |
| Passengers 12 months earlier | known (it falls before T) | **not known**: it falls inside the forecast window | used only as the year-1 anchor |
| Passengers in the same month of the year before T | known | known | **anchor for both years** (the seasonal naive) |
| Our own forecasts fed back as lags | errors compound | errors compound | **not used**: direct, one-shot forecast |
| Route, airports, distance, country, coordinates | fixed | fixed | **keep** |
| Calendar month | known | known | **keep** |
| Which routes will exist in the window | not known | not known | forecast set fixed at T from routes seen in the prior 12 months |
| `Unnamed: 0`, `SF_LOAD_TMS` | export artefacts | export artefacts | **drop** |
| Settings picked after seeing the test score | — | — | **not allowed**: all settings live in `src/forecasting.py` |

In [ ]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd

MODELING = next(p for p in [Path.cwd(), *Path.cwd().parents, Path.cwd() / "team-10-modeling"]
                if (p / "src" / "forecasting.py").exists())
sys.path.insert(0, str(MODELING / "src"))

from data import BANNER, load_airports, load_pax, write_table
from features import (DROP_PAX, FEATURES, KEY, MODEL_CATEGORICAL, MODEL_NUMERIC, airport_table,
                      clean_pax, model_inputs, route_table, training_pairs)
from forecasting import forecast_set, origins, seasonal_naive

print(BANNER)
raw = load_pax()

## 1. How strongly does same-month revenue track the target?

In [ ]:
r = raw[["OD_PAX", "OD_REV_USD"]].dropna()
print(f"corr(OD_PAX, OD_REV_USD) = {r.corr().iloc[0, 1]:.3f}; "
      f"on log1p scale = {np.log1p(r.clip(lower=0)).corr().iloc[0, 1]:.3f}")

High by construction: revenue is passengers × average fare. A model given same-month revenue would
look excellent and be useless, because revenue for a future month does not exist at T. Dropped
(`DROP_PAX` in `src/features.py`). Past revenue could be a lagged input, but it adds little beyond
past passengers and inherits the same horizon limits, so it is not used.

## 2. Build the feature frame

In [ ]:
pax = clean_pax(raw)
print("dropped columns:", [c for c in DROP_PAX if c in raw.columns])
print(f"rows after cleaning: {len(pax):,} (removed {len(raw) - len(pax):,})")
routes = route_table(pax, airport_table(load_airports()))
o = origins(pax)
print("origins:", {k: str(v.date()) for k, v in o.__dict__.items()})

train = training_pairs(pax[pax["DATE"] < o.test], routes, o.test)
fs = forecast_set(pax, o.test)
rows = model_inputs(fs.assign(anchor=seasonal_naive(pax, fs, o.test)), routes)
print(f"training pairs before the test origin: {len(train):,}")
print(f"forecast set at the test origin: {len(fs):,} route-months "
      f"({fs[KEY].drop_duplicates().shape[0]:,} routes x 24 months)")

In [ ]:
assert set(FEATURES) <= set(train.columns) and set(FEATURES) <= set(rows.columns)
assert train["DATE"].max() < o.test, "a training pair reaches past the origin"
frame = pd.DataFrame({
    "kind": ["numeric"] * len(MODEL_NUMERIC) + ["categorical"] * len(MODEL_CATEGORICAL),
    "pct_missing_train": [round(100 * train[c].isna().mean(), 2) for c in FEATURES],
    "pct_missing_forecast": [round(100 * rows[c].isna().mean(), 2) for c in FEATURES],
}, index=FEATURES)
frame

Missing coordinates or runway values come from codes missing in the airport lookup. The Ridge
pipeline imputes the median; the tree model handles missing values natively. Neither invents a
country or state.

**Feature set handed to D4:** `MODEL_NUMERIC + MODEL_CATEGORICAL` from `src/features.py`, with
target `log_growth = log(passengers / same month a year earlier)`. The per-feature rationale is in
`results/feature-set.md`.

In [ ]:
md = ["| feature | kind | % missing (training pairs) | % missing (test forecast set) |", "|---|---|---|---|"]
md += [f"| {f} | {r.kind} | {r.pct_missing_train} | {r.pct_missing_forecast} |" for f, r in frame.iterrows()]
print("wrote", write_table("feature_frame.md",
      "# Feature frame — Team 10 (D3)\n\n_Generated by `notebooks/03_leakage_and_features.ipynb`._\n\n"
      f"- Training pairs before the test origin ({o.test.date()}): {len(train):,}\n"
      f"- Forecast set at the test origin: {len(fs):,} route-months\n\n" + "\n".join(md) + "\n"))